# Foreground Tracker

> Track the active Windows foreground application and persist completed foreground and idle intervals.

This module reads foreground-window and process metadata, applies the configured
idle thresholds, and identifies changes in the application, window, or title
receiving the user's attention.

The monitoring loop converts repeated samples into intervals rather than storing
every sample. Completed intervals are written to the database when foreground
state changes or monitoring stops, with prolonged inactivity recorded as an
`IDLE` interval.

In [ ]:
#| default_exp foreground_tracker

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import psutil
from datetime import datetime
from time import sleep

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| export
import snooper_pkg.config as cf
from snooper_pkg.db import *
from snooper_pkg.idle_detector import *

ModuleNotFoundError: No module named 'snooper_pkg.idle_detector'

## Foreground window inspection

In [ ]:
#| export
import sys
def _require_windows():
    if sys.platform != "win32":
        raise RuntimeError("Windows idle detection is only available on Windows")


In [ ]:
#| export
def get_foreground_window_info() -> tuple[str, int, int, str]:
    "Return the foreground window title, handle, process ID, and process name."
    _require_windows()
    import win32gui, win32process
    hwnd = win32gui.GetForegroundWindow()
    title = win32gui.GetWindowText(hwnd) or "Unknown"
    tid, pid = win32process.GetWindowThreadProcessId(hwnd)
    if not isinstance(pid, int) or pid <= 0: return title, hwnd, -1, "UNKNOWN"
    try: process_name = psutil.Process(pid).name()
    except (psutil.NoSuchProcess, psutil.AccessDenied, ValueError, psutil.Error): return title, hwnd, -1, "UNKNOWN"
    return title, hwnd, pid, process_name

## Streaming-specific idle thresholds


In [ ]:
#| export
def is_streaming_app(
    title: str,  # Foreground window title to inspect
    streaming_app_min_idle_time: dict[str, int | float],  # Title substrings mapped to idle thresholds in seconds
) -> int | float | None:
    "Return the idle threshold for the first configured title match, or `None`."
    for s in streaming_app_min_idle_time.keys():
        if s.lower() in title.lower():
            return streaming_app_min_idle_time[s]
    return None

## Foreground interval monitoring


In [ ]:
#| export
def start_foreground_app_monitoring(
    session_id: str,  # Monitoring session receiving foreground intervals
    stop_event: object,  # Stop signal providing an `is_set()` method
    time_interval: int | float = cf.FG_APP_MONITORING_INTERVAL_SECONDS,  # Seconds between foreground samples
    min_idle_time_gap: int | float = cf.FG_APP_MONITORING_MIN_IDLE_SECONDS,  # General idle threshold in seconds
    streaming_app_min_idle_time: dict[str, int | float] = cf.STREAMING_APP_MIN_IDLE_SECONDS,  # Title-specific idle thresholds
) -> None:
    "Sample foreground state until stopped and record each completed interval."
    fg_dict = {}
    try:
        while not stop_event.is_set():
            title, hwnd, pid, process_name = get_foreground_window_info()
            if is_streaming_app(title, streaming_app_min_idle_time) is not None: 
                min_idle_time = is_streaming_app(title, streaming_app_min_idle_time)
                is_idle = get_idle_seconds() > min_idle_time
            else: is_idle = get_idle_seconds() > min_idle_time_gap
            if is_idle:
                title, hwnd, pid, process_name = "Idle", -1, -1, "IDLE"
            if fg_dict.get("pid") != pid or fg_dict.get("hwnd") != hwnd or fg_dict.get("window_title") != title:
                if fg_dict:
                    fg_dict["end_time"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                    log_fg_app_events(**fg_dict)
                fg_dict = {
                    "session_id":session_id,
                    "window_title": title,
                    "pid": pid,
                    "hwnd": hwnd,
                    "app": process_name,
                    "start_time": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                    "is_idle":is_idle,
                    "end_time": None
                }
            else:
                fg_dict["end_time"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            sleep(time_interval)
    except KeyboardInterrupt:
        print("Stopped by user.")
    finally:
        if fg_dict: 
            fg_dict["end_time"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            log_fg_app_events(**fg_dict)


NameError: name 'cf' is not defined

- `stop_event` is documented only as an object providing `is_set()`. The code
  does not establish whether it must specifically be a `threading.Event` or may
  be another compatible stop signal.

- `is_streaming_app` returns the first case-insensitive substring match.
  Therefore dictionary insertion order determines the result when several
  configured strings match the same title. Confirm that this priority rule is
  intentional.

- `start_foreground_app_monitoring` calls `is_streaming_app` twice whenever a
  streaming-title match is found. This is redundant but has been preserved to
  avoid changing behaviour.

- Raw foreground window titles are passed to `log_fg_app_events`. The current
  implementation does not consult `SHOW_WINDOW_TITLES` or apply title-cleaning
  or privacy rules. Confirm that retaining raw titles is acceptable for the
  present MVP.

- `KeyboardInterrupt` is caught inside the monitoring function. If this
  function normally runs in a worker thread, confirm whether that exception
  handling is expected to be effective there.


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()